# Semana 4: Ciclo de Vida de los Datos con Pandas

En esta clase recorremos las etapas centrales del trabajo con datos usando **Pandas** y **NumPy**:
desde la creación de un dataset reproducible, pasando por la limpieza e imputación, el análisis
temporal, hasta la visualización y el storytelling.

---

## Temario

| Bloque | Tema | Herramientas principales |
|--------|------|--------------------------|
| 1 | Configuración y datos reproducibles | `pd.DataFrame`, `np.random.default_rng` |
| 2 | Limpieza e imputación manual | `.str`, `.fillna()`, `np.where`, `.loc` |
| 3 | Análisis de series temporales | `.resample()`, `.rolling()`, `.shift()` |
| 4 | Visualización y storytelling | `matplotlib`, `plotly` |
| 2.5 | Pipeline automatizado (nivel profesional) | `sklearn.Pipeline`, `SimpleImputer` |

> **Tip**: cada bloque construye sobre el anterior. Ejecutá las celdas en orden.

# Bloque 1: Configuración y Creación de Datos Reproducibles
**Temas: 1, 8, 9**

---

## ¿Por qué datos sintéticos?

Antes de analizar datos reales, los data scientists suelen trabajar con **datos sintéticos** —
datos generados por código que imitan la realidad. Las ventajas:

- **Reproducibilidad**: cualquier persona del equipo obtiene exactamente el mismo dataset.
- **Control total**: conocemos los valores correctos y podemos verificar que el código funciona.
- **Independencia**: no necesitamos esperar que el cliente nos envíe los datos.

## La semilla aleatoria (`seed`)

Cuando pedimos números "al azar", la computadora sigue una fórmula matemática que parte de un
número inicial llamado **semilla**. Con la misma semilla → mismo resultado siempre.

```python
rng = np.random.default_rng(42)  # 42 es la semilla; cualquier número sirve
```

En equipos de trabajo, fijar la semilla garantiza que todos vean los mismos datos aunque
corran el notebook en momentos distintos.

## Estructura del dataset que vamos a crear

Simulamos las transacciones de una tienda online durante 100 días:

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `id_transaccion` | entero | identificador único |
| `fecha` | datetime | una venta por día |
| `cliente` | string (sucio) | nombre con espacios de más |
| `monto` | float | valor de la venta en USD |
| `categoria` | string / NaN | tipo de producto (10% nulos) |

> Intencionalmente dejamos datos sucios (espacios, nulos) para practicar limpieza en el Bloque 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# ── Generador de números aleatorios reproducible ──────────────────────
# default_rng(42) reemplaza al antiguo np.random.seed(). Es más moderno
# y seguro para entornos de trabajo colaborativo.
rng = np.random.default_rng(42)
n = 100  # cantidad de transacciones a simular

# ── Construcción del diccionario de datos ─────────────────────────────
data = {
    # IDs consecutivos del 1 al n
    'id_transaccion': range(1, n+1),

    # pd.date_range genera una secuencia de fechas; freq='D' = un día por fila
    'fecha': pd.date_range(start='2026-01-01', periods=n, freq='D'),

    # Simulamos datos SUCIOS: los nombres tienen espacios adelante y atrás
    # Esto es muy común en datos reales exportados de planillas de Excel
    'cliente': [f'  Usuario_{i}  ' for i in rng.integers(1, 20, n)],

    # Montos entre $10 y $500 con distribución uniforme
    'monto': rng.uniform(10, 500, n),

    # rng.choice permite asignar probabilidades: 30% cada categoría, 10% NaN
    'categoria': rng.choice(
        ['Electrónica', 'Hogar', 'Moda', np.nan],
        n,
        p=[0.3, 0.3, 0.3, 0.1]  # las probabilidades deben sumar 1.0
    )
}

# ── Creación del DataFrame ─────────────────────────────────────────────
# pd.DataFrame convierte el diccionario en una tabla de filas y columnas
df = pd.DataFrame(data)

# ── Introducción de nulos adicionales en 'monto' ──────────────────────
# Elegimos 10 filas al azar y les asignamos NaN para simular datos faltantes
df.loc[rng.choice(n, 10, replace=False), 'monto'] = np.nan

print(f"Dataset creado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Nulos por columna:\n{df.isnull().sum()}")
print("\nPrimeras filas:")
df.head()

## Ejercicio Bloque 1: "Cambiando la escala del negocio"

**Objetivo**: entender cómo la semilla y el tamaño de la muestra afectan los datos.

1. Cambiá el valor de `n` a `500`.
2. Cambiá la semilla `42` por tu número de la suerte.
3. Volvé a ejecutar la celda y observá el resultado.

**Pregunta técnica**: ¿Los montos siguen siendo los mismos que antes?
¿Por qué es importante fijar la semilla en un equipo de trabajo?

---

# Bloque 2: Limpieza e Imputación
**Temas: 2, 3, 10, 11, 12**

---

## ¿Por qué limpiar los datos?

Los datos reales casi nunca llegan perfectos. Los problemas más comunes son:
- **Espacios ocultos** en texto: `"  Juan "` vs `"Juan"` son strings distintos.
- **Mayúsculas inconsistentes**: `"JUAN"`, `"juan"` y `"Juan"` son tres valores diferentes.
- **Valores nulos (NaN)**: filas donde el dato no existe o no fue registrado.

Si no limpiamos, los filtros, los joins y los modelos de machine learning van a fallar
o a dar resultados incorrectos.

---

## 1. Limpieza de Strings (Normalización de texto)

```python
df['cliente'] = df['cliente'].str.strip().str.title()
```

El accesor **`.str`** permite aplicar funciones de texto a toda una columna de una sola vez
(sin usar un bucle `for`). Esto se llama **vectorización** y es mucho más rápido.

| Método | Qué hace | Ejemplo |
|--------|----------|---------|
| `.str.strip()` | Elimina espacios al inicio y al final | `"  Lucho "` → `"Lucho"` |
| `.str.title()` | Primera letra mayúscula, resto minúscula | `"LUCHO"` → `"Lucho"` |
| `.str.upper()` | Todo en mayúsculas | `"Lucho"` → `"LUCHO"` |
| `.str.lower()` | Todo en minúsculas | `"Lucho"` → `"lucho"` |

---

## 2. Detección y Tratamiento de Nulos (Imputación)

**Imputar** significa rellenar los valores faltantes con un valor calculado.
¿Con qué valor? Depende del tipo de dato:

| Estrategia | Cuándo usarla | Función |
|------------|---------------|---------|
| **Mediana** | Números con valores extremos (outliers) | `df['col'].median()` |
| **Media** | Números sin outliers, distribución normal | `df['col'].mean()` |
| **Moda** | Texto o categorías | `df['col'].mode()[0]` |
| **Constante** | Cuando sabemos el valor correcto | `df['col'].fillna(0)` |

> **¿Por qué mediana y no promedio para `monto`?**  
> Si una venta fue de $50.000 (un outlier), el promedio sube mucho pero la mediana no se mueve.
> La mediana es más representativa del cliente típico.

---

## 3. Transformaciones Vectorizadas: `np.where`

```python
df['segmento_venta'] = np.where(df['monto'] > 300, 'Alta', 'Normal')
```

Es el equivalente a la función `SI()` de Excel, pero aplicada a miles de filas
a la vez sin ningún `for`. Sintaxis: `np.where(condición, valor_si_verdadero, valor_si_falso)`.

---

## 4. Filtrado Avanzado con `.loc` y máscaras booleanas

```python
df_moda = df.loc[(df['categoria'] == 'Moda') & (df['monto'] > 100)]
```

- **`.loc`**: selecciona filas y columnas por **etiqueta** (nombre). Es el selector estándar en pandas.
- **Máscara booleana**: cada condición genera una columna de `True`/`False`.
- **Operador `&`** (AND): exige que **ambas** condiciones sean verdaderas.
- **Operador `|`** (OR): alcanza con que **una** condición sea verdadera.
- Siempre poner cada condición entre paréntesis `( )`.

In [ ]:
# ── 1. Limpieza de strings ────────────────────────────────────────────
# .str.strip() elimina espacios invisibles; .str.title() estandariza mayúsculas
# Encadenamos los dos métodos con el punto (method chaining)
df['cliente'] = df['cliente'].str.strip().str.title()

print("Ejemplo de nombre limpio:", df['cliente'].iloc[0])

# ── 2. Imputación de valores nulos ────────────────────────────────────
# Calculamos la mediana ANTES de imputar (si imputáramos primero, la mediana cambiaría)
mediana_monto = df['monto'].median()
print(f"\nMediana del monto (para imputar): ${mediana_monto:.2f}")

# fillna() reemplaza NaN con el valor que le pasamos
df['monto'] = df['monto'].fillna(mediana_monto)

# Para texto usamos la moda: el valor que más aparece
# mode() devuelve una Serie; con [0] tomamos el primero (el más frecuente)
categoria_mas_frecuente = df['categoria'].mode()[0]
print(f"Moda de categoría (para imputar): {categoria_mas_frecuente}")
df['categoria'] = df['categoria'].fillna(categoria_mas_frecuente)

# ── 3. Transformación vectorizada: np.where ───────────────────────────
# Creamos una nueva columna clasificando cada venta según su monto
# np.where(condición, valor_si_True, valor_si_False)
df['segmento_venta'] = np.where(df['monto'] > 300, 'Alta', 'Normal')

# ── 4. Filtrado con .loc y máscaras booleanas ─────────────────────────
# Solo filas donde categoría es 'Moda' Y monto es mayor a 100
# Los paréntesis en cada condición son obligatorios cuando usamos & o |
df_moda = df.loc[(df['categoria'] == 'Moda') & (df['monto'] > 100)]

# ── Verificación final ────────────────────────────────────────────────
print(f"\nNulos restantes en todo el DataFrame: {df.isnull().sum().sum()}")
print(f"\nDistribución de segmentos de venta:\n{df['segmento_venta'].value_counts()}")
print(f"\nVentas de Moda > $100: {len(df_moda)} transacciones")

## Ejercicio Bloque 2: "Criterio de Segmentación"

**Objetivo**: manipular la lógica condicional y la limpieza de texto.

1. En la limpieza de strings, usá `.str.upper()` en lugar de `.str.title()`
   para que todos los nombres queden en **MAYÚSCULAS**.
2. Modificá el `np.where`: las ventas se consideran `'VIP'` si el monto es
   mayor a `400`, y `'Estándar'` si no.

**Pregunta técnica**: Al ejecutar `.value_counts()`, ¿cuántos clientes VIP
tenemos ahora en comparación con el código original?

---

In [ ]:
# Solución ejercicio Bloque 2

# 1. Normalización en MAYÚSCULAS
df['cliente'] = df['cliente'].str.strip().str.upper()

# 2. Nueva lógica de segmentación VIP (umbral más alto: > 400)
df['segmento_venta'] = np.where(df['monto'] > 400, 'VIP', 'Estándar')

# Contamos cuántos clientes caen en cada segmento
print("Distribución del nuevo segmento:")
print(df['segmento_venta'].value_counts())

# Bloque 3: Análisis Temporal (Series de Tiempo)
**Temas: 5, 6, 7**

---

## ¿Qué es una Serie de Tiempo?

Es cualquier secuencia de datos donde **el orden importa** y cada registro tiene una fecha
o marca de tiempo asociada. Ejemplos: precio de una acción, temperatura diaria,
ventas por hora.

Para que Pandas trate la fecha como el eje principal del análisis, la convertimos en
el **índice** del DataFrame:

```python
df = df.set_index('fecha')
```

---

## Resampling: cambiar la granularidad temporal

**Resamplear** es agrupar datos a una frecuencia diferente. Es como alejar o acercar
la cámara: podés ver el día a día o el resumen mensual del mismo dataset.

```python
df['monto'].resample('W').sum()   # suma semanal
df['monto'].resample('ME').sum()  # suma mensual
```

| Código | Frecuencia | Descripción |
|--------|------------|-------------|
| `'D'`  | Diario | Un valor por día |
| `'W'`  | Semanal | Agrupa de domingo a sábado |
| `'ME'` | Mensual | Agrupa por fin de mes |
| `'QE'` | Trimestral | Agrupa por trimestre |
| `'YE'` | Anual | Agrupa por año |
| `'2W'` | Quincenal | Combinación: número + letra |

---

## Rolling: ventana deslizante (suavizado)

```python
df['monto'].rolling(window=7).mean()
```

A diferencia del resampling, aquí **no cambiamos la frecuencia**. Para cada día,
miramos los 6 días anteriores y promediamos los 7. Esto elimina el "ruido" de un
día atípico y revela la **tendencia real** del negocio.

> Los primeros 6 valores quedarán como `NaN` porque no tienen suficiente historia.

---

## Shift: desplazamiento temporal (lag)

```python
df['monto'].shift(1)  # desplaza 1 posición hacia abajo
```

Sirve para comparar un valor con el del período anterior:

```python
df['variacion'] = df['monto'] - df['monto'].shift(1)
```

Esto nos dice: *¿cuánto subió o bajó la venta respecto a ayer?*

In [ ]:
# ── 1. Establecer el índice temporal ─────────────────────────────────
# Pandas necesita que la columna de fecha SEA el índice para habilitar
# las operaciones temporales como resample() y rolling()
if 'fecha' in df.columns:
    # pd.to_datetime garantiza que la columna sea de tipo datetime64
    # (a veces viene como string si la leímos de un CSV)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.set_index('fecha')
    print("Índice temporal establecido:", df.index.dtype)
else:
    print("La columna 'fecha' no existe o ya es el índice.")

# ── 2. Resampling: agregar datos a frecuencia semanal ─────────────────
# resample('W') agrupa por semana; .sum() suma los montos de cada semana
ventas_semanales = df['monto'].resample('W').sum()
print(f"\nSemanas en el dataset: {len(ventas_semanales)}")
print(ventas_semanales.head())

# ── 3. Rolling: media móvil de 7 días ─────────────────────────────────
# window=7: usa los 7 días anteriores (incluyendo el actual) para calcular la media
# min_periods=1 (no lo usamos aquí) evitaría los NaN iniciales si quisiéramos
df['monto_suavizado'] = df['monto'].rolling(window=7).mean()

# ── 4. Shift: variación respecto al día anterior ───────────────────────
# shift(1) mueve todos los valores UNA posición hacia abajo;
# la diferencia entre hoy y ayer nos da la variación diaria
df['dif_dia_anterior'] = df['monto'] - df['monto'].shift(1)

print("\nComparación monto diario vs suavizado (primeras 10 filas):")
print(df[['monto', 'monto_suavizado']].head(10).round(2))

# ── 5. Gráfico: ruido diario vs tendencia suavizada ───────────────────
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['monto'], label='Venta Diaria (ruido)', alpha=0.4, color='gray')
plt.plot(df.index, df['monto_suavizado'], label='Media Móvil 7 días (tendencia)', color='blue', linewidth=2)
plt.title('Ventas Diarias vs. Tendencia Suavizada')
plt.xlabel('Fecha')
plt.ylabel('Monto ($)')
plt.legend()
plt.tight_layout()
plt.show()

# ── 6. Gráfico de barras: ventas totales por semana ───────────────────
plt.figure(figsize=(10, 4))
ventas_semanales.plot(kind='bar', color='orange', alpha=0.7)
plt.title('Ventas Totales por Semana (Resampling semanal)')
plt.ylabel('Monto Total ($)')
plt.xlabel('Semana')
plt.tight_layout()
plt.show()

## Ejercicio Bloque 3: "Cambiando la mirada temporal"

**Objetivo**: entender la diferencia entre cambiar la frecuencia y cambiar la ventana.

1. Cambiá `resample('W')` por `resample('ME')` para ver el total mensual.
2. Cambiá `window=7` del rolling por `window=15`.

**Preguntas técnicas**:
- ¿La línea de tendencia azul se volvió más suave o más "nerviosa" con `window=15`?
- ¿Cuántas barras aparecen ahora en el gráfico de ventas? ¿Por qué son menos?

---

In [ ]:
# Solución ejercicio Bloque 3

# 1. Resampling mensual (ME = Month End)
# Ahora cada barra del gráfico representará un mes entero en lugar de una semana
ventas_mensuales = df['monto'].resample('ME').sum()

# 2. Media móvil con ventana más larga (15 días)
# Una ventana mayor suaviza más pero pierde sensibilidad a cambios recientes
df['monto_suavizado_15d'] = df['monto'].rolling(window=15).mean()

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel izquierdo: suavizado original (7d) vs nuevo (15d)
axes[0].plot(df.index, df['monto'], alpha=0.3, color='gray', label='Original')
axes[0].plot(df.index, df['monto_suavizado'], color='blue', alpha=0.6, label='Rolling 7d')
axes[0].plot(df.index, df['monto_suavizado_15d'], color='red', label='Rolling 15d')
axes[0].set_title('Comparación: ventana 7d vs 15d')
axes[0].legend()

# Panel derecho: ventas mensuales (menos barras, vista más agregada)
ventas_mensuales.plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.7)
axes[1].set_title('Ventas Totales por Mes')
axes[1].set_ylabel('Monto Total ($)')

plt.tight_layout()
plt.show()

# Bloque 4: Visualización y Storytelling con Datos
**Temas: 13, 14, 15, 16**

---

## ¿Qué es el Storytelling con datos?

Un análisis que nadie entiende no sirve. El **storytelling** es la habilidad de
comunicar hallazgos de datos de forma clara, convincente y accionable para
quien toma decisiones (que generalmente no es técnico).

Los tres componentes del storytelling con datos:
1. **Datos correctos**: el análisis debe ser riguroso.
2. **Visualización efectiva**: el gráfico correcto para el mensaje correcto.
3. **Narrativa**: contexto, causa, consecuencia y recomendación.

---

## Matplotlib vs Plotly

| Característica | Matplotlib | Plotly |
|----------------|------------|--------|
| Tipo | Estático (imagen) | Interactivo (HTML) |
| Uso ideal | Reportes PDF, papers | Dashboards, presentaciones |
| Curva de aprendizaje | Media | Baja |
| Zoom / hover | No | Sí |

---

## Herramientas clave

**`plt.hist`**: muestra la distribución de los datos.
¿Las ventas se concentran en montos bajos o altos? ¿La distribución es simétrica?

**`px.line` (Plotly)**: gráfico de línea interactivo. El usuario puede hacer zoom,
ver el valor exacto al pasar el mouse y exportar como imagen.

**`fig.add_annotation`**: agregar etiquetas sobre el gráfico para señalar
eventos importantes (lanzamiento de campaña, cambio de precio, etc.).
Aquí es donde pasás de ser programador a ser consultor de negocio.

In [ ]:
# ── 1. Histograma con Matplotlib ─────────────────────────────────────
# El histograma divide el rango de valores en 'bins' y cuenta cuántas
# ventas caen en cada rango. Nos muestra la DISTRIBUCIÓN de los montos.
plt.figure(figsize=(10, 4))
plt.hist(df['monto'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribución de los Montos de Venta')
plt.xlabel('Monto (USD)')
plt.ylabel('Cantidad de transacciones')
# Una distribución uniforme (como la nuestra) tiene barras de altura similar;
# en datos reales verías concentraciones en ciertos rangos de precio
plt.tight_layout()
plt.show()

# ── 2. Gráfico interactivo con Plotly ─────────────────────────────────
# reset_index() convierte el índice (fecha) de vuelta a columna para que Plotly lo use
fig = px.line(
    df.reset_index(),
    x='fecha',
    y='monto_suavizado',
    title='Tendencia de Ventas 2026: Evolución Suavizada',
    labels={'monto_suavizado': 'Ventas (Media Móvil 7d)', 'fecha': 'Fecha'},
    template='plotly_white'  # fondo blanco; otras opciones: 'plotly_dark', 'ggplot2'
)

# ── 3. Anotación estratégica (storytelling) ───────────────────────────
# Marcamos el punto máximo de ventas para que el lector lo note de inmediato
# showarrow=True + arrowhead=1 dibuja una flecha apuntando al dato
fig.add_annotation(
    x='2026-04-01',
    y=df['monto_suavizado'].max(),
    text="Punto máximo del trimestre",
    showarrow=True,
    arrowhead=2,
    bgcolor='lightyellow',
    bordercolor='black'
)

fig.show()

In [ ]:
# Personalización del eje X: mostrar etiquetas mensuales
# Por defecto Plotly elige la granularidad del eje X automáticamente.
# update_xaxes() nos da control total sobre el formato.

fig2 = px.line(
    df.reset_index(),
    x='fecha',
    y='monto_suavizado',
    title='Evolución Mensual de Ventas (eje X personalizado)'
)

fig2.update_xaxes(
    dtick='M1',            # intervalo entre etiquetas: 1 mes
    tickformat='%b %Y',    # formato: 'Ene 2026', 'Feb 2026', etc.
    ticklabelmode='period' # centra la etiqueta en el período
)

# Añadimos línea de referencia del promedio para dar contexto
promedio_suavizado = df['monto_suavizado'].mean()
fig2.add_hline(
    y=promedio_suavizado,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Promedio: ${promedio_suavizado:.0f}'
)

fig2.show()

## Ejercicio Bloque 4: "Personalización del Storytelling"

**Objetivo**: practicar la personalización de gráficos interactivos.

1. Cambiá el `template` de `'plotly_white'` a `'plotly_dark'`.
2. Agregá una segunda anotación señalando el **punto mínimo** del dataset
   (pista: usá `.min()` y buscá la fecha con `.idxmin()`).

**Pregunta técnica**: ¿Cómo cambia la percepción del gráfico con fondo oscuro?
¿En qué tipo de presentación usarías cada template?

---

In [ ]:
# Solución ejercicio Bloque 4

df_temp = df.copy()

# Buscamos el punto medio del dataset para la anotación principal
posicion_media = len(df_temp) // 2
fecha_media = df_temp.index[posicion_media]
valor_media = df_temp['monto_suavizado'].iloc[posicion_media]

# idxmin() devuelve el ÍNDICE (fecha) donde el valor es mínimo
# Usamos dropna() porque los primeros 6 días tienen NaN por el rolling
fecha_min = df_temp['monto_suavizado'].dropna().idxmin()
valor_min = df_temp['monto_suavizado'].loc[fecha_min]

# Gráfico en modo oscuro con dos anotaciones
fig3 = px.line(
    df_temp.reset_index(),
    x='fecha',
    y='monto_suavizado',
    title='Reporte de Tendencia – Modo Oscuro',
    template='plotly_dark'
)

# Anotación 1: evento de marketing en el punto medio
fig3.add_annotation(
    x=fecha_media, y=valor_media,
    text='Evento de Marketing',
    showarrow=True, arrowhead=1, arrowcolor='yellow', font=dict(color='yellow')
)

# Anotación 2: punto mínimo (insight negativo a investigar)
fig3.add_annotation(
    x=fecha_min, y=valor_min,
    text='Valle: ¿qué pasó aquí?',
    showarrow=True, arrowhead=2, arrowcolor='tomato', font=dict(color='tomato')
)

fig3.show()

# Ejercicio Integrador: "El Semáforo de Ventas"

Este ejercicio combina todo lo visto en los bloques anteriores en un mini-proyecto
de principio a fin.

**Objetivo**: limpiar el dataset y crear una visualización que comunique
de forma inmediata qué ventas son buenas y cuáles necesitan atención.

**Pasos**:
1. **Limpieza**: eliminar espacios en los nombres de clientes.
2. **Imputación**: rellenar montos faltantes con el promedio del grupo.
3. **Clasificación**: crear columna `estado` → `'Festejar'` si monto > 300, sino `'Mejorar'`.
4. **Visualización**: gráfico de barras donde el color refleja el estado.

> Este flujo (limpiar → transformar → visualizar) es exactamente lo que
> harías con datos reales de un cliente.

In [ ]:
# ── Paso 1: Limpieza de texto ─────────────────────────────────────────
# str.strip() porque puede haber espacios residuales de pasos anteriores
df['cliente'] = df['cliente'].str.strip()

# ── Paso 2: Imputación con el promedio ────────────────────────────────
# Usamos mean() (promedio) en lugar de median() para mostrar la alternativa
# En datos sin outliers el promedio es aceptable
promedio = df['monto'].mean()
print(f"Promedio de monto (para imputar): ${promedio:.2f}")
df['monto'] = df['monto'].fillna(promedio)

# ── Paso 3: Clasificación condicional ─────────────────────────────────
# np.where(condición, valor_si_verdadero, valor_si_falso)
df['estado'] = np.where(df['monto'] > 300, 'Festejar', 'Mejorar')
print(f"\nDistribución del estado:\n{df['estado'].value_counts()}")

# ── Paso 4: Visualización interactiva ─────────────────────────────────
# color_discrete_map: mapeamos manualmente cada categoría a un color
# Verde para ventas buenas, rojo para las que necesitan atención
fig4 = px.bar(
    df.reset_index(),
    x='id_transaccion',
    y='monto',
    color='estado',
    title='Semáforo de Ventas: ¿Qué transacciones destacar?',
    labels={'monto': 'Monto ($)', 'id_transaccion': 'ID Transacción'},
    color_discrete_map={'Festejar': 'green', 'Mejorar': 'red'}
)

# Agregamos una línea de referencia en el umbral de clasificación
fig4.add_hline(y=300, line_dash='dash', line_color='black',
               annotation_text='Umbral: $300')

fig4.show()

# Bloque 2.5: Automatización con Scikit-Learn (Nivel Profesional)

---

## ¿Por qué automatizar la limpieza?

Hasta ahora hicimos la limpieza **manualmente**, paso a paso. Esto funciona para
explorar, pero tiene un problema grave en producción:

> ¿Qué pasa cuando llegan datos nuevos mañana? ¿Y la semana que viene?

Necesitamos un **proceso repetible** que no dependa de que alguien recuerde
ejecutar cada paso en el orden correcto.

## Las tres piezas del rompecabezas

### 1. `SimpleImputer` – el que rellena nulos
```python
SimpleImputer(strategy='median')  # rellena con la mediana
SimpleImputer(strategy='most_frequent')  # rellena con la moda
```
Aprende la mediana/moda del dataset de **entrenamiento** y luego la aplica
a datos nuevos sin recalcularla. Esto evita el **data leakage** (filtración de información
del futuro al pasado).

### 2. `ColumnTransformer` – el director de orquesta
```python
ColumnTransformer(transformers=[
    ('num', imputer_mediana, columnas_numericas),
    ('cat', imputer_moda, columnas_categoricas),
])
```
Aplica **diferentes transformaciones a diferentes columnas** en un solo paso.
Columnas numéricas con mediana, categóricas con moda.

### 3. `Pipeline` – la cinta transportadora
```python
pipeline = Pipeline(steps=[
    ('imputacion', preprocessor),
    ('modelo', LinearRegression()),
])
```
El dato entra sucio por un extremo y sale transformado (y predicho) por el otro.
Un solo `.fit()` y un solo `.predict()`, sin importar cuántos pasos haya en el medio.

| Ventaja | Descripción |
|---------|-------------|
| Reproducibilidad | El proceso es siempre el mismo |
| Sin data leakage | La transformación aprende solo del train set |
| Desplegable | Se puede serializar con `joblib.dump()` |
| Grid Search | Compatible con `GridSearchCV` para tuneo de hiperparámetros |

In [ ]:
# ==========================================
# BLOQUE 2.5: Pipeline de Imputación Profesional
# ==========================================
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── Preparación del DataFrame ──────────────────────────────────────────
# Trabajamos sobre una copia para no modificar el df que venimos usando
# reset_index() devuelve 'fecha' a columna (deja de ser índice)
df_ml = df.copy().reset_index()

# Separamos columnas por tipo de dato
# Importante: el imputer numérico no puede procesar texto, y viceversa
num_cols = ['monto']
cat_cols = ['categoria']

# ── Sub-pipelines por tipo de columna ─────────────────────────────────
# Cada Pipeline encadena pasos; aquí solo hay imputación, pero
# se podría agregar StandardScaler u otros transformadores
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))  # mediana para números
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))  # moda para texto
])

# ── ColumnTransformer: aplica cada sub-pipeline a sus columnas ─────────
# remainder='passthrough' = mantener las columnas que NO transformamos
# sin esto, las columnas no especificadas desaparecerían del resultado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ],
    remainder='passthrough'
)

# ── fit_transform: aprender Y aplicar en un solo paso ─────────────────
# fit() → calcula mediana/moda del dataset
# transform() → aplica esas estadísticas para rellenar nulos
# fit_transform() = ambos a la vez (solo para el conjunto de entrenamiento)
datos_array = preprocessor.fit_transform(df_ml)

# ── Convertir de vuelta a DataFrame ───────────────────────────────────
# sklearn devuelve un array de NumPy; lo convertimos para mayor legibilidad
# El orden de columnas es: primero las transformadas, luego las passthrough
columnas_finales = num_cols + cat_cols + [
    c for c in df_ml.columns if c not in num_cols + cat_cols
]
df_final = pd.DataFrame(datos_array, columns=columnas_finales)

print("Pipeline ejecutado exitosamente.")
print(f"Nulos restantes: {df_final.isnull().sum().sum()}")
print("\nPrimeras filas del resultado:")
print(df_final.head())